# P1-L3｜信号证据审计

> **对应课程**：第 9–16 课  
> **模块**：P1-M3 · 统计证据与可预测性  
> **数据身份**：固定随机种子模拟教学数据，不是真实证券行情。

同时检查效应大小、显著性、时间依赖、样本外表现和多重检验边界。

本实验向最终策略报告提供：**统计证据、效应大小与样本外验证**。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 按时间切分统计证据

同一信号同时检查效应大小、t 检验、市场 Beta 和样本外方向；不把一个 p 值当作结论。

In [2]:
import statsmodels.api as sm
from scipy import stats

daily = pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
close = daily.pivot(index="date", columns="symbol", values="adj_close").sort_index()
returns = close.pct_change(fill_method=None)
signal = close["SIM_GROWTH"].pct_change(20).shift(1)
strategy = np.sign(signal) * returns["SIM_GROWTH"]
dataset = pd.concat([
    strategy.rename("strategy"),
    returns["SIM_MARKET"].rename("market"),
], axis=1).dropna()

split = int(len(dataset) * 0.7)
train, test = dataset.iloc[:split], dataset.iloc[split:]
test_result = stats.ttest_1samp(test["strategy"], 0.0)
model = sm.OLS(test["strategy"], sm.add_constant(test[["market"]])).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

audit = pd.Series({
    "train_mean_bps": train["strategy"].mean() * 10_000,
    "test_mean_bps": test["strategy"].mean() * 10_000,
    "test_pvalue": test_result.pvalue,
    "test_beta": model.params["market"],
    "test_alpha_bps": model.params["const"] * 10_000,
    "test_observations": len(test),
})
display(audit.to_frame("value"))

,value
train_mean_bps,1.912592
test_mean_bps,19.143403
test_pvalue,0.052737
test_beta,0.218019
test_alpha_bps,18.809718
test_observations,221.000000


## 验收

- 关键计算能由小例子或独立公式核对；
- 数据时点、信号时点和成交时点没有混用；
- 结果同时包含金融含义和失败边界；
- 产物能够进入第 29 课策略报告和第 30 课能力矩阵。